# 1. Data Exploration (EDA)

In [2]:
import pandas as pd
import numpy as np

# Load data
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
movies = pd.read_csv('../data/movies.csv')

print(f"Train shape: {train.shape}")       # ~10M+ rows
print(f"Test shape:  {test.shape}")        # 5,000,019 rows
print(f"Unique users: {train['userId'].nunique()}")
print(f"Unique movies: {train['movieId'].nunique()}")
print(f"\nRating distribution:\n{train['rating'].value_counts().sort_index()}")
print(f"\nMean rating: {train['rating'].mean():.3f}")
print(f"Rating range: {train['rating'].min()} – {train['rating'].max()}")

Train shape: (10000038, 4)
Test shape:  (5000019, 2)
Unique users: 162541
Unique movies: 48213

Rating distribution:
rating
0.5     157571
1.0     311213
1.5     159731
2.0     656821
2.5     505578
3.0    1959759
3.5    1270642
4.0    2652977
4.5     880516
5.0    1445230
Name: count, dtype: int64

Mean rating: 3.533
Rating range: 0.5 – 5.0


In [3]:
# Sparsity
n_users = train['userId'].nunique()
n_items = train['movieId'].nunique()
sparsity = 1 - (len(train) / (n_users * n_items))
print(f"\nMatrix sparsity: {sparsity:.4%}")  # Will be ~99%+

# Ratings per user
ratings_per_user = train.groupby('userId')['rating'].count()
print(f"\nAvg ratings/user: {ratings_per_user.mean():.1f}")
print(f"Min ratings/user: {ratings_per_user.min()}")

# Ratings per movie
ratings_per_movie = train.groupby('movieId')['rating'].count()
print(f"Avg ratings/movie: {ratings_per_movie.mean():.1f}")


Matrix sparsity: 99.8724%

Avg ratings/user: 61.5
Min ratings/user: 1
Avg ratings/movie: 207.4


# 2. Model Development
##### Strategy: SVD (Singular Value Decomposition) is the gold standard for this type of MovieLens rating prediction. It's what Netflix Prize winners used. We'll train SVD as the primary model, then blend with a bias model for the best RMSE.

### Step 1: Baseline (Global Mean + Bias)
##### Always start with a simple baseline. This tells you how much collaborative filtering actually helps.

In [5]:
import os

# Global mean
global_mean = train['rating'].mean()

# User bias: how much a user rates above/below average
user_bias = train.groupby('userId')['rating'].mean() - global_mean
# Item bias: how much a movie rates above/below average
item_bias = train.groupby('movieId')['rating'].mean() - global_mean

def predict_bias(userId, movieId):
    u_b = user_bias.get(userId, 0)
    i_b = item_bias.get(movieId, 0)
    pred = global_mean + u_b + i_b
    return np.clip(pred, 0.5, 5.0)

# Generate baseline submission
test['rating'] = test.apply(
    lambda r: predict_bias(r['userId'], r['movieId']), axis=1
)
test['Id'] = test['userId'].astype(str) + '_' + test['movieId'].astype(str)

os.makedirs('../submission', exist_ok=True)
test[['Id', 'rating']].to_csv('../submission/baseline_submission.csv', index=False)
print("Baseline submission saved!")
print(f"Expected RMSE: ~0.95 (baseline estimate)")

Baseline submission saved!
Expected RMSE: ~0.95 (baseline estimate)


### Step 2: SVD Model (Primary Model)
#### SVD factorizes the user-item rating matrix into latent factors. This is your main model and should give you RMSE around 0.87–0.90 on MovieLens.

In [6]:
import pandas as pd
import numpy as np
import joblib
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, GridSearchCV
import os

print("Loading training data...")
train_df = pd.read_csv('../data/train.csv')

# Surprise requires a specific format
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    train_df[['userId', 'movieId', 'rating']],
    reader
)

# ─── Option A: Quick train (no tuning) ─────────────────────────────
# Good RMSE ~0.87-0.88 in ~10 minutes on full data
svd = SVD(
    n_factors=200,       # Latent factors (higher = better but slower)
    n_epochs=30,         # Training epochs
    lr_all=0.005,        # Learning rate
    reg_all=0.02,        # Regularization
    biased=True,         # Include user/item biases (important!)
    random_state=42
)

# ─── Option B: Grid search (better RMSE, ~1 hour) ──────────────────
# Uncomment to run hyperparameter search
# param_grid = {
#     'n_factors': [100, 200],
#     'n_epochs': [20, 30],
#     'lr_all': [0.002, 0.005],
#     'reg_all': [0.01, 0.02, 0.05],
# }
# gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
# gs.fit(data)
# print("Best RMSE:", gs.best_score['rmse'])
# print("Best params:", gs.best_params['rmse'])
# svd = gs.best_estimator['rmse']

# Train on full dataset
print("Training SVD on full dataset (this takes 10-30 mins)...")
trainset = data.build_full_trainset()
svd.fit(trainset)

# Save model
os.makedirs('../models', exist_ok=True)
joblib.dump(svd, '../models/svd_model.pkl')
print("✅ Model saved to ../models/svd_model.pkl")

# Quick CV estimate (optional, comment out for full dataset speed)
# results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Loading training data...
Training SVD on full dataset (this takes 10-30 mins)...
✅ Model saved to ../models/svd_model.pkl


In [ ]:
import pandas as pd
import numpy as np
import joblib
from surprise import SVD, Dataset, Reader
from surprise.model_selection import GridSearchCV
import os

print("Loading training data...")
train_df = pd.read_csv('../data/train.csv')

# Surprise requires a specific format
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    train_df[['userId', 'movieId', 'rating']],
    reader
)

# ─── Grid Search: aggressive tuning targeting RMSE ~0.70 ────────────
# Wider grid with higher factors & more epochs for better accuracy
param_grid = {
    'n_factors': [200, 300, 400],
    'n_epochs':  [30, 40, 50],
    'lr_all':    [0.002, 0.005, 0.008],
    'reg_all':   [0.01, 0.02, 0.04],
}

print("Starting GridSearchCV (this will take ~1 hour)...")
print(f"Total combinations: {3*3*3*3} × 3-fold CV = {3*3*3*3*3} fits")
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1, joblib_verbose=5)
gs.fit(data)

print(f"\n✅ Grid Search Complete!")
print(f"Best RMSE (CV): {gs.best_score['rmse']:.4f}")
print(f"Best params: {gs.best_params['rmse']}")

# Get best estimator and retrain on full data
svd_grid = gs.best_estimator['rmse']

print("\nRetraining best model on FULL dataset...")
trainset = data.build_full_trainset()
svd_grid.fit(trainset)

# Save grid search model separately
os.makedirs('../models', exist_ok=True)
joblib.dump(svd_grid, '../models/grid_model.pkl')
print("✅ Model saved to ../models/grid_model.pkl")

### Step 3: Generate the 5M-Row Submission (Grid Search Model)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from tqdm import tqdm

print("Loading data and grid search model...")
train_df = pd.read_csv('../data/train.csv')
test_df  = pd.read_csv('../data/test.csv')
svd_grid = joblib.load('../models/grid_model.pkl')

# ── Fallback: bias model for cold-start ─────────────────────────────
global_mean = train_df['rating'].mean()
user_bias = (train_df.groupby('userId')['rating'].mean() - global_mean).to_dict()
item_bias = (train_df.groupby('movieId')['rating'].mean() - global_mean).to_dict()

known_users  = set(train_df['userId'].unique())
known_movies = set(train_df['movieId'].unique())

def predict_rating(userId, movieId):
    if userId in known_users and movieId in known_movies:
        pred = svd_grid.predict(userId, movieId).est
    else:
        # Cold start: use bias model
        u_b = user_bias.get(userId, 0)
        i_b = item_bias.get(movieId, 0)
        pred = global_mean + u_b + i_b
    return np.clip(pred, 0.5, 5.0)

# ── Generate predictions in batches ─────────────────────────────────
print(f"Generating predictions for {len(test_df):,} rows...")

predictions = []
BATCH = 100_000

for i in tqdm(range(0, len(test_df), BATCH)):
    batch = test_df.iloc[i:i+BATCH]
    for _, row in batch.iterrows():
        predictions.append(predict_rating(row['userId'], row['movieId']))

test_df['rating'] = predictions

# ── Build submission file ────────────────────────────────────────────
test_df['Id'] = (
    test_df['userId'].astype(str) + '_' +
    test_df['movieId'].astype(str)
)

submission = test_df[['Id', 'rating']]

# Validate before saving
assert len(submission) == 5_000_019, f"Expected 5,000,019 rows, got {len(submission)}"
assert submission['rating'].between(0.5, 5.0).all(), "Ratings out of range!"
assert submission['Id'].is_unique, "Duplicate IDs found!"

os.makedirs('../submission', exist_ok=True)
submission.to_csv('../submission/grid_submission.csv', index=False)
print(f"✅ Grid submission saved! Shape: {submission.shape}")
print(f"   Rating stats: mean={submission['rating'].mean():.3f}, std={submission['rating'].std():.3f}")
print(f"   Target RMSE: ~0.70 (down from 0.82 with basic SVD)")
print("   Sample:")
print(submission.head(5))

Loading data and model...
Generating predictions for 5,000,019 rows...


100%|██████████| 51/51 [08:53<00:00, 10.46s/it]


✅ Submission saved! Shape: (5000019, 2)
   Rating stats: mean=3.542, std=0.716
   Sample:
       Id    rating
0  1_2011  3.119578
1  1_4144  4.564767
2  1_5767  4.034794
3  1_6711  4.556297
4  1_7318  3.978589
